# PROD: BLUCK 0-shot (Bengali-only) — Qwen3.5-9B Q4_K_M GGUF via llama-server (T4)

**Model:** `psxog/SAHAYAKAI` / `Qwen3.5-9B.Q4_K_M.gguf` (~5.78GB Q4_K_M)

**Task:** BLUCK 0-shot, Bengali-only: 2366 MCQs, 23 subcategories in 4 domains (History / Culture / Phonetics / Semantics) from `https://github.com/minhaj1403/bluck` (`bn_dataset/**/*.csv`, columns `question,a,b,c,d,answer`). No CoT, no 5-shot examples.

**Prompt (Bengali-only, paper-faithful):** instruction + `প্রশ্ন / A-D / উত্তর:` — model must output only the option letter. Same intent as paper Fig 3(a) system+user prompt.

**Backend:** `llama-server` (`base_url=http://127.0.0.1:8081`, **completion mode, NO chat template**) — same as BenHallu/MGSM prod notebooks.

**Decoding:** greedy (`temperature=0.0, top_p=1.0`), `max_tokens=16` (letter needs ~2 tokens).

**Parsing (KoBBQ-style, paper §4.1):** single A-D → `উত্তর: X` → option-text substring → blank/error. Blanks count as wrong.

**Metric:** accuracy overall + per-domain + per-subcategory. Paper 0-shot baselines: GPT-4o 0.727, Claude-3.5-Sonnet 0.735, Gemini-1.5-Pro 0.617, Llama-3.3-70B 0.554, DeepSeekV3 0.693.

**Outputs:** `/content/bluck_qwen3p5_9b_0shot.jsonl` (2366, resume-safe) + `/content/bluck_per_id.csv` + `/content/bluck_results.json` + `/content/bluck_comparison.csv`. Progress via `tqdm` bar.

## 1. Setup — Install & GPU check

In [ ]:
!nvidia-smi
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 llama-cpp-python
!pip install -q huggingface_hub pandas tqdm requests matplotlib
# Fallback if cu121 404s:
# !pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 llama-cpp-python
!curl -LsSf https://llama.app/install.sh | sh
!export PATH="$HOME/.local/bin:$PATH"; ~/.local/bin/llama --version
!python -c "import llama_cpp; print('GPU offload:', llama_cpp.llama_supports_gpu_offload())"
import sys
print('Python', sys.version)

## 2. Download model (`psxog/SAHAYAKAI` Q4_K_M)

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo_id = 'psxog/SAHAYAKAI'
filename = 'Qwen3.5-9B.Q4_K_M.gguf'
print(f'Downloading {repo_id}/{filename} ...')
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f'Model: {model_path} ({os.path.getsize(model_path)/1e9:.2f} GB)')
os.environ['MODEL_PATH'] = model_path
%env MODEL_PATH={model_path}

## 3. Start llama-server on 8081 (completion-mode backend)

In [ ]:
import os
print('MODEL_PATH:', os.environ.get('MODEL_PATH'))
# 8080 is often held in Colab, so use 8081.
!pkill -9 -f llama 2>/dev/null; sleep 2; echo 'killed old server'
!export PATH="$HOME/.local/bin:$PATH"; nohup ~/.local/bin/llama serve -m "$MODEL_PATH" -c 4096 --port 8081 -ngl -1 --parallel 1 > /tmp/llama8081.log 2>&1 &
!sleep 10; curl -s http://127.0.0.1:8081/health; echo; tail -n 20 /tmp/llama8081.log
# Expect first: 503 Loading model (normal, 9B takes 30-90s).

## 4. Wait until server ready

In [ ]:
!for i in $(seq 1 12); do echo -n "$i: "; curl -s http://127.0.0.1:8081/health; echo; sleep 10; done
!tail -n 10 /tmp/llama8081.log
# Proceed only when a poll returns {"status":"ok"}.

## 5. Smoke-test completions endpoint (must return choices before eval)

In [ ]:
!curl -s -X POST http://127.0.0.1:8081/v1/completions -H 'Content-Type: application/json' -d '{"prompt":"1+1=","max_tokens":5}' | head -c 300; echo

## 6. Dataset — clone BLUCK, merge 23 CSVs → ITEMS (expect 2366)

In [ ]:
!rm -rf /content/bluck && git clone --depth 1 https://github.com/minhaj1403/bluck /content/bluck
!git -C /content/bluck rev-parse HEAD; echo '---'; ls /content/bluck/bn_dataset/*/ | head -n 40

import csv, json, os, re
from pathlib import Path
from collections import Counter

BASE = Path('/content/bluck/bn_dataset')
csvs = sorted(BASE.rglob('*.csv'))
print(f'CSV files: {len(csvs)} (expect 23)')
for p in csvs:
    print(' ', p.relative_to(BASE))

def parse_gold(raw):
    # Tolerant: 'A' -> ['A']; 'A, C' / 'A/C' / 'a c' -> ['A','C']; '' / 'X' -> []
    if raw is None:
        return [], ''
    s = str(raw).strip().upper()
    parts = [p for p in re.split(r'[^A-D]+', s) if p in ('A', 'B', 'C', 'D')]
    # de-dup preserve order
    seen = []
    for p in parts:
        if p not in seen:
            seen.append(p)
    return seen, s

ITEMS, BAD_ROWS, MULTI_ROWS = [], [], []
for p in csvs:
    domain = p.parent.name  # Culture / History / Phonetics / Semantics
    subcat = p.stem        # e.g. Geography, Synonyms
    with open(p, encoding='utf-8-sig', newline='') as f:
        for i, r in enumerate(csv.DictReader(f)):
            gold_set, gold_raw = parse_gold(r.get('answer'))
            rec = {
                'id': f'{domain}_{subcat}_{i:04d}',
                'domain': domain, 'subcategory': subcat,
                'question': (r.get('question') or '').strip(),
                'a': (r.get('a') or '').strip(), 'b': (r.get('b') or '').strip(),
                'c': (r.get('c') or '').strip(), 'd': (r.get('d') or '').strip(),
                'gold_raw': gold_raw, 'gold_set': gold_set,
                'gold': gold_set[0] if gold_set else '',
                'is_multi': len(gold_set) > 1,
            }
            if not gold_set:
                BAD_ROWS.append({**rec, 'src': str(p), 'row': i})
                continue
            if rec['is_multi']:
                MULTI_ROWS.append({**rec, 'src': str(p), 'row': i})
            ITEMS.append(rec)

print(f'ITEMS={len(ITEMS)} BAD={len(BAD_ROWS)} MULTI={len(MULTI_ROWS)} TOTAL={len(ITEMS)+len(BAD_ROWS)} (expect 2366)')
assert len(ITEMS) + len(BAD_ROWS) == 2366, f'BLUCK count changed: {len(ITEMS)+len(BAD_ROWS)}'
if MULTI_ROWS:
    print(f'MULTI-ANSWER rows ({len(MULTI_ROWS)}): strict=first-letter, lenient=in-set. Examples:')
    for m in MULTI_ROWS[:10]:
        print(f"  {m['id']} gold_raw={m['gold_raw']!r} gold_set={m['gold_set']} Q={m['question'][:80]!r}")
if BAD_ROWS:
    print(f'BAD rows skipped ({len(BAD_ROWS)}):')
    for b in BAD_ROWS[:10]:
        print(f"  {b['id']} gold_raw={b['gold_raw']!r} src={b['src']}:{b['row']}")
import pandas as _pd
if MULTI_ROWS:
    _pd.DataFrame(MULTI_ROWS).to_csv('/content/bluck_multi_audit.csv', index=False)
    print(f"saved /content/bluck_multi_audit.csv ({len(MULTI_ROWS)} rows)")
if BAD_ROWS:
    _pd.DataFrame(BAD_ROWS).to_csv('/content/bluck_bad_skipped.csv', index=False)
    print(f"saved /content/bluck_bad_skipped.csv ({len(BAD_ROWS)} rows)")
print('by domain:', dict(Counter(x['domain'] for x in ITEMS)))
print('by subcategory:')
for k, v in sorted(Counter((x['domain'], x['subcategory']) for x in ITEMS).items()):
    print(f'  {k[0]}/{k[1]}: {v}')
print(json.dumps(ITEMS[0], ensure_ascii=False)[:600])

## 7. Prompt (Bengali-only) + `extract_label` + server client

Completion mode: raw Bengali prompt POSTed to `/v1/completions` — no chat template, no system role, no English.

In [ ]:
import re, time
import requests

BASE_URL = 'http://127.0.0.1:8081'

# Bengali-only prompt. Only non-Bengali chars are the A/B/C/D option markers (required by the task).
PROMPT_TEMPLATE = """আপনি বাংলা ভাষা ও সংস্কৃতি বিষয়ক বিশেষজ্ঞ। নিচের বহুনির্বাচনী প্রশ্নের সঠিক উত্তরের শুধুমাত্র অপশন অক্ষর দিন (A, B, C, D)। কোনো ব্যাখ্যা দেবেন না।

প্রশ্ন: {question}
A) {a}
B) {b}
C) {c}
D) {d}
উত্তর:"""

def build_prompt(it):
    return PROMPT_TEMPLATE.format(question=it['question'], a=it['a'], b=it['b'], c=it['c'], d=it['d'])

def strip_think_tags(text):
    return re.sub(r'<think>.*?</think>', '', text or '', flags=re.DOTALL).strip()

def extract_label(raw, it=None):
    """KoBBQ-style, paper section 4.1: letter-only > উত্তর:/Answer: > option-text mention. Else '' (blank=error)."""
    if raw is None:
        return ''
    t = strip_think_tags(raw).strip()
    if not t:
        return ''
    # 1) standalone A-D on first line (most common greedy output)
    m = re.search(r'^\s*\(?([A-D])\)?[\).:\-\s]*$', t.splitlines()[0].strip(), re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # 2) উত্তর: / Answer: / Option X patterns anywhere in head
    head = '\n'.join(t.splitlines()[:3])[:300]
    m = re.search(r'(?:উত্তর|Answer|Option)\s*[:\-]?\s*\(?([A-D])\)?', head, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # 3) bare leading letter e.g. 'C' or 'C.'
    m = re.search(r'^\s*([A-D])\b', t, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # 4) option-text substring fallback (only if full option text appears verbatim)
    if it:
        for L in ('A', 'B', 'C', 'D'):
            opt = (it[L.lower()] or '').strip()
            if opt and len(opt) >= 2 and opt in t:
                return L
    return ''

def complete(prompt, max_tokens=16, timeout=300):
    r = requests.post(f'{BASE_URL}/v1/completions',
                      json={'prompt': prompt, 'max_tokens': max_tokens,
                            'temperature': 0.0, 'top_p': 1.0, 'stream': False},
                      timeout=timeout)
    r.raise_for_status()
    return r.json()['choices'][0]['text']

# Quick test on 1 item per domain
import random
random.seed(0)
seen = set()
for it in ITEMS:
    if it['domain'] not in seen:
        seen.add(it['domain'])
        raw = complete(build_prompt(it))
        print(f"{it['domain']}/{it['subcategory']} gold={it['gold']} gold_raw={it.get('gold_raw', it['gold'])} raw={raw[:80]!r} parsed={extract_label(raw, it)!r}")

## 8. Run eval — full 2366 (greedy, resume-safe, tqdm progress bar, 1 retry on blank)

In [ ]:
import json, os, time
from tqdm.auto import tqdm

OUT = '/content/bluck_qwen3p5_9b_0shot.jsonl'
done = {}
if os.path.exists(OUT):
    with open(OUT, encoding='utf-8') as f:
        for line in f:
            try:
                d = json.loads(line)
                if d.get('parsed') in ('A', 'B', 'C', 'D'):
                    done[d['id']] = d['parsed']
            except Exception:
                pass
    print(f'Resuming: {len(done)} already judged A-D')

t0 = time.time(); n_new = n_retry = 0
pbar = tqdm(total=len(ITEMS), initial=len(done), unit='q', desc='BLUCK 0-shot (bn)')
with open(OUT, 'a', encoding='utf-8') as fout:
    for i, it in enumerate(ITEMS):
        if it['id'] in done:
            continue
        prompt = build_prompt(it)
        try:
            raw = complete(prompt)
        except Exception as e:
            raw = ''
            print(f"  [HTTP-ERROR] id={it['id']} {e}")
        parsed = extract_label(raw, it)
        if not parsed:
            time.sleep(1); n_retry += 1
            try:
                raw2 = complete(prompt)
            except Exception:
                raw2 = ''
            if extract_label(raw2, it):
                raw, parsed = raw2, extract_label(raw2, it)
        gs = it.get('gold_set') or ([it['gold']] if it.get('gold') else [])
        fout.write(json.dumps({
            'id': it['id'], 'domain': it['domain'], 'subcategory': it['subcategory'],
            'gold': it['gold'], 'gold_raw': it.get('gold_raw', it['gold']), 'gold_set': gs,
            'is_multi': bool(it.get('is_multi', False)),
            'raw_response': raw, 'parsed': parsed,
            'correct': int(parsed == it['gold']),
            'correct_lenient': int(bool(parsed) and parsed in gs),
            'decoding': {'temperature': 0.0, 'top_p': 1.0, 'max_tokens': 16,
                         'mode': 'completion', 'chat_template': False, 'shot': '0-shot', 'lang': 'bn'}
        }, ensure_ascii=False) + '\n')
        fout.flush(); n_new += 1; pbar.update(1)
        if n_new % 25 == 0:
            pbar.set_postfix({'new': n_new, 'retries': n_retry, 'min': f"{(time.time()-t0)/60:.1f}"})
        if (i + 1) % 200 == 0:
            print(f"{i+1}/{len(ITEMS)} new={n_new} retries={n_retry} elapsed={(time.time()-t0)/60:.1f} min")
pbar.close()
print(f'Done in {(time.time()-t0)/60:.1f} min new={n_new} retries={n_retry} -> {OUT}')
!wc -l /content/bluck_qwen3p5_9b_0shot.jsonl

## 9. Score — overall + per-domain + per-subcategory + paper comparison

In [ ]:
import json, pandas as pd
from collections import Counter

recs = [json.loads(l) for l in open('/content/bluck_qwen3p5_9b_0shot.jsonl', encoding='utf-8')]
print(f'records: {len(recs)} (expect len(ITEMS)={len(ITEMS)}, total=2366)')
df = pd.DataFrame(recs)
df['correct'] = (df['parsed'] == df['gold']).astype(int)
if 'correct_lenient' not in df.columns:
    df['correct_lenient'] = df.apply(lambda r: int(bool(r['parsed']) and r['parsed'] in (r.get('gold_set') or [r['gold']])), axis=1)
if 'is_multi' not in df.columns:
    df['is_multi'] = False
n_blank = int((~df['parsed'].isin(['A', 'B', 'C', 'D'])).sum())
n_multi = int(df['is_multi'].sum())

overall = float(df['correct'].mean())
overall_lenient = float(df['correct_lenient'].mean())
df_single = df[~df['is_multi']]
overall_single = float(df_single['correct'].mean()) if len(df_single) else 0.0
print(f'OVERALL strict (paper-comparable) = {overall:.4f} ({int(df["correct"].sum())}/{len(df)})')
print(f'OVERALL single-only = {overall_single:.4f} ({int(df_single["correct"].sum())}/{len(df_single)})  MULTI={n_multi}')
print(f'OVERALL lenient (in-set) = {overall_lenient:.4f}  blank/unparseable = {n_blank}')
print('\nPer-domain:')
dom = df.groupby('domain')['correct'].agg(['mean', 'sum', 'count']).round(4)
print(dom.to_string())
print('\nPer-subcategory:')
sub = df.groupby(['domain', 'subcategory'])['correct'].agg(['mean', 'sum', 'count']).round(4).sort_values('mean')
print(sub.to_string())

# Paper Table 2 overall 0-shot baselines
paper = {'GPT-4o': 0.727, 'Claude-3.5-Sonnet': 0.735, 'Gemini-1.5-Pro': 0.617,
         'Llama-3.3-70B-Instruct': 0.554, 'DeepSeekV3': 0.693}
rows = [{'model': m, 'acc_0shot': a} for m, a in paper.items()]
rows.append({'model': 'Qwen3.5-9B SAHAYAKAI Q4_K_M (this run, 0-shot bn)', 'acc_0shot': round(overall, 4)})
cmp_df = pd.DataFrame(rows).sort_values('acc_0shot', ascending=False)
print('\n' + cmp_df.to_string(index=False))
cmp_df.to_csv('/content/bluck_comparison.csv', index=False)
df.to_csv('/content/bluck_per_id.csv', index=False)

import subprocess
sha = subprocess.run(['git', '-C', '/content/bluck', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
summary = {
    'model': 'psxog/SAHAYAKAI Qwen3.5-9B.Q4_K_M.gguf',
    'backend': 'llama-server GGUF Q4_K_M on T4 (completion, no chat template)',
    'decoding': {'temperature': 0.0, 'top_p': 1.0, 'max_tokens': 16},
    'protocol': 'BLUCK 0-shot Bengali-only, no CoT',
    'prompt': 'Bengali-only: প্রশ্ন + A-D + উত্তর: (letter-only instruction)',
    'parsing': 'KoBBQ-style: letter > উত্তর:/Answer: > option-text > blank=error',
    'data': {'repo': 'https://github.com/minhaj1403/bluck', 'sha': sha, 'n': len(df),
             'n_total_expected': 2366, 'n_multi': n_multi, 'n_bad_skipped': 2366 - len(ITEMS)},
    'overall_accuracy': overall,
    'overall_single_only': overall_single, 'overall_lenient': overall_lenient,
    'n_correct': int(df['correct'].sum()), 'n_correct_lenient': int(df['correct_lenient'].sum()),
    'n_blank': n_blank,
    'by_domain': {k: {'acc': float(v['mean']), 'n': int(v['count'])} for k, v in dom.iterrows()},
    'by_subcategory': {f'{a}/{b}': {'acc': float(v['mean']), 'n': int(v['count'])}
                       for (a, b), v in sub.iterrows()},
}
json.dump(summary, open('/content/bluck_results.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print(open('/content/bluck_results.json', encoding='utf-8').read()[:2000])

try:
    import matplotlib.pyplot as plt
    dom.plot.barh(y='mean', legend=False, title='BLUCK 0-shot per-domain accuracy (Qwen3.5-9B)')
    plt.tight_layout(); plt.savefig('/content/bluck_domains.png', dpi=120)
    print('saved /content/bluck_domains.png')
except Exception as e:
    print('plot skipped:', e)

## 10. Check outputs

In [ ]:
import os
for p in ['/content/bluck_qwen3p5_9b_0shot.jsonl', '/content/bluck_results.json',
          '/content/bluck_per_id.csv', '/content/bluck_comparison.csv',
          '/content/bluck_multi_audit.csv', '/content/bluck_bad_skipped.csv']:
    print(('OK ' if os.path.exists(p) else 'MISS ') + p + (f' ({os.path.getsize(p)/1e3:.1f} KB)' if os.path.exists(p) else ''))
# from google.colab import files
# for p in ['/content/bluck_results.json', '/content/bluck_comparison.csv']:
#     files.download(p)